# 张量的典型应用

## 1.标量

In [6]:
import tensorflow as tf
import keras.datasets.imdb

layers = tf.keras.layers

# 以均方差误差函数为例，经过 tf.keras.losses.mse(或 tf.keras.losses.MSE，两者相同功能)
# 返回每个样本上的误差值，最后取误差的均值作为当前 Batch 的误差，它是一个标量
out = tf.random.uniform([4, 10])  # 随机模拟网络输出
y = tf.constant([2, 3, 1, 8])  # 随机构造样品真实标签
y = tf.one_hot(y, depth=10)
loss = tf.keras.losses.mse(y, out)
print('loss:', loss)
loss = tf.reduce_mean(loss)
print('loss_reduce_mean:', loss)

loss: tf.Tensor([0.37407535 0.40938443 0.34075508 0.20459597], shape=(4,), dtype=float32)
loss_reduce_mean: tf.Tensor(0.3322027, shape=(), dtype=float32)


## 2.向量

In [8]:
# 向量是一种非常常见的数据载体，如在全连接层和卷积神经网络层中，偏置张量𝒃就使用向量来表示.
# z = w * x, 模拟获得激活函数的输入Z
z = tf.random.normal([4, 2])
b = tf.zeros([2])
z = z + b
print('b:', b)
print('z:', z)

b: tf.Tensor([0. 0.], shape=(2,), dtype=float32)
z: tf.Tensor(
[[ 0.5513485   1.3726989 ]
 [-0.8490585   2.0030446 ]
 [-0.92311144 -0.6785197 ]
 [-0.86465865  0.5794217 ]], shape=(4, 2), dtype=float32)


In [ ]:
#  可以通过全连接层的 bias 成员变量查看偏置变量 𝒃 ，例如创建输入节点数为 4， 输出节点数为 3 的线性层网络，那么它的偏置向量 b 的长度应为 3.
fc = layers.Dense(3)  # 创建一层wx + b，输出节点为3
# 通过build函数创建w,b张量，输入节点为4
fc.build(input_shape=(2, 4))
fc.bias

<tf.Variable 'bias:0' shape=(3,) dtype=float32, numpy=array([0., 0., 0.], dtype=float32)>

## 3.矩阵

矩阵也是非常常见的张量类型，比如全连接层的批量输入张量𝑿的形状为[𝑏, 𝑑in]，其
中𝑏表示输入样本的个数，即 Batch Size，𝑑in表示输入特征的长度。

In [13]:
x = tf.random.normal([2, 4])  # 2个样本，特征长度为4的张量。
# 令全连接层的输出节点数为3
w = tf.ones([4, 3])
b = tf.zeros([3])
o = x @ w + b
print('o:', o)

o: tf.Tensor(
[[-0.07145962 -0.07145962 -0.07145962]
 [-2.5818214  -2.5818214  -2.5818214 ]], shape=(2, 3), dtype=float32)


其中𝑿和𝑾张量均是矩阵，上述代码实现了一个线性变换的网络层，激活函数为空。一般地，𝜎(𝑿@𝑾 + 𝒃)网络层称为全连接层，在 TensorFlow 中可以通过 Dense 类直接实现，特别地，**当激活函数𝜎为空时，全连接层也称为线性层**。  
我们通过 Dense 类创建输入 4 个节点，输出 3 个节点的网络层，并通过全连接层的 kernel 成员名查看其权值矩阵.

In [ ]:
fc = layers.Dense(3)  # 定义全连接层的输出节点为 3
fc.build(input_shape=(2, 4))  # 定义全连接层的输入节点为 4
fc.kernel  # 查看权值矩阵 W

<tf.Variable 'kernel:0' shape=(4, 3) dtype=float32, numpy=
array([[-0.5127734 ,  0.3751787 , -0.14954793],
       [-0.5335776 , -0.16680616, -0.13329762],
       [-0.6984458 ,  0.13965356, -0.61140025],
       [-0.39349735, -0.57339054,  0.79977465]], dtype=float32)>

## 4.三维张量

三维的张量一个典型应用是表示序列信号，它的格式是
𝑿 = [𝑏, sequence len, feature len]  
其中𝑏表示序列信号的数量，sequence len 表示序列信号在时间维度上的采样点数或步数，
feature len 表示每个点的特征长度。

考虑自然语言处理(Natural Language Processing，简称 NLP)中句子的表示，如评价句
子的是否为正面情绪的情感分类任务网络，如图 4.3 所示。为了能够方便字符串被神经网
络处理，一般将单词通过*嵌入层(Embedding Layer)编码为固定长度的向量*，比如“a”编码
为某个长度 3 的向量，那么 2 个等长(单词数量为 5)的句子序列可以表示为 shape 为[2,5,3]
的 3 维张量，其中 2 表示句子个数，5 表示单词数量，3 表示单词向量的长度。

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=10000)  # 自动加载IMDB电影评价数据集
# num_words=10000，数据集的大小不变（训练集、测试集还是 25,000 条影评），只保留词频排名 1 ~ 9999 的单词（因为 0、1、2 通常被预留给特殊符号），
# 排名在 10,000 以外的低频词会被替换成一个特殊的“未知词符号”（通常是索引 2，表示 <UNK>）
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=80)
x_train.shape

(25000, 80)

可以看到 x_train 张量的 shape 为[25000,80]，其中 25000 表示句子个数，80 表示每个句子
共 80 个单词，每个单词使用数字编码方式表示。  
我们通过 layers.Embedding 层将数字编码的*单词*转换为长度为 100 个词向量：

In [ ]:
embedding = layers.Embedding(10000, 100)  # 创建词向量 Embedding 层类
out = embedding(x_train)  # 将数字编码的单词转换为词向量
out.shape

TensorShape([25000, 80, 100])

## 5.四维张量

四维张量在卷积神经网络中应用非常广泛，它用于保存特征图(Feature maps)数据，格
式一般定义为
[𝑏, ℎ, , 𝑐]
其中𝑏表示输入样本的数量，ℎ/ 分别表示特征图的高/宽，𝑐表示特征图的通道数，部分深
度学习框架也会使用[𝑏, 𝑐, ℎ, ]格式的特征图张量，例如 PyTorch

In [ ]:
# 创建32x32的彩色图片输入，数量为4
x = tf.random.normal([4, 32, 32, 3])
# 创建卷积神经网络
layer = layers.Conv2D(16, kernel_size=3)
out = layer(x)  # 向前计算
out.shape

TensorShape([4, 30, 30, 16])

其中卷积核也是4维张量，可以通过kernel成员变量访问

In [31]:
layer.kernel.shape

TensorShape([3, 3, 3, 16])